# Data

> Data classes and functions

In [1]:
#| default_exp data

In [2]:
#| hide
from nbdev.showdoc import *

## Data types

In [3]:
#| export
import os

from bioMONAI.core import MetaTensor, torchTensor, BypassNewMeta, DisplayedTransform, torchsqueeze, Path, List, L, torchmax, randint, typedispatch
from bioMONAI.io import img_reader
from bioMONAI.visualize import show_images_grid



### Meta resolver

In [4]:
#| export

class MetaResolver(type(torchTensor), metaclass=BypassNewMeta):
    """
    A class to bypass metaclass conflict:
    https://pytorch-geometric.readthedocs.io/en/latest/_modules/torch_geometric/data/batch.html
    """
    pass
    

### BioImageBase

In [5]:

#| export
class BioImageBase(MetaTensor, metaclass=MetaResolver):
    """
    A class that represents an image object.
    Metaclass casts `x` to this class if it is of type `cls._bypass_type`.
    """
    
    _bypass_type = torchTensor  # The type that bypasses image loading
    _show_args = {'cmap': 'gray'}  # Default arguments for image display
    resample, reorder = None, False  # Default resample and reorder settings
    affine_matrix = None  # Default affine matrix for image transformation

    @classmethod
    def create(cls, fn: (Path, str, List, torchTensor), **kwargs) -> torchTensor: 
        """
        Opens an image and casts it to BioImageBase object.
        If `fn` is a torchTensor, it's cast to BioImageBase object.

        Args:
            fn : (Path, str, torchTensor)
                Image path or a 4D torchTensor.
            kwargs : dict
                Additional parameters for the medical image reader.

        Returns:
            torchTensor : A 4D tensor as a BioImageBase object.
        """
        if isinstance(fn, torchTensor):
            return cls(fn)

        return img_reader(fn, dtype=cls, resample=cls.resample, reorder=cls.reorder)

    @classmethod
    def item_preprocessing(cls, resample: (List, int, tuple), reorder: bool):
        """
        Changes the values for the class variables `resample` and `reorder`.

        Args:
            resample : (List, int, tuple)
                A list with voxel spacing.
            reorder : bool
                Whether to reorder the data to be closest to canonical (RAS+) orientation.
        """
        cls.resample = resample
        cls.reorder = reorder

    def show(self, ctx=None, figsize: int = None, ncols: int = 10, **kwargs):
        """
        Plots 2D slices of a 3D image alongside a prior specified axis.

        Args:
            ctx : Context to use for the display. Defaults to None.
            figsize: Size of the figure. Defaults to None.
            ncols: Number of columns in the grid. Defaults to 10.
            **kwargs : Additional keyword arguments passed to plt.imshow.

        Returns:
            Shown image.
        """
        return show_images_grid(self, ctx=ctx, ncols=ncols, **merge(self._show_args, kwargs))
    
    def as_tensor(self) -> torchTensor:
        """
        Return the `MetaTensor` as a `torchTensor`.
        It is OS dependent as to whether this will be a deep copy or not.
        """
        return self.as_subclass(torchTensor)

    def __repr__(self) -> str:
        """Returns the string representation of the ImageBase instance."""
        return f"BioImageBase{self.as_tensor().__repr__()[6:]}"

### BioImage 

class for 2D Images

In [6]:
#| export

class BioImage(BioImageBase):
    """Subclass of BioImageBase that represents 2D and 3D image objects."""
    _show_args = {'cmap':'gray'}
    
    @classmethod
    def create(cls, fn: (Path, str, L, list, torchTensor), **kwargs) -> torchTensor: 
        """
        Opens an image and casts it to BioImageBase object.
        If `fn` is a torchTensor, it's cast to BioImageBase object.

        Args:
            fn : (Path, str, torchTensor)
                Image path or a 4D torchTensor.
            kwargs : dict
                Additional parameters for the medical image reader.

        Returns:
            torchTensor : A 3D tensor as a BioImage object.
        """
        if isinstance(fn, torchTensor):
            return cls(fn)

        return torchsqueeze(img_reader(fn, dtype=cls, resample=cls.resample, reorder=cls.reorder), 1)
    
    def show(self, ctx=None, **kwargs):
        "Show image using `merge(self._show_args, kwargs)`"
        return show_image(self, ctx=ctx, **merge(self._show_args, kwargs))
    
    def __repr__(self) -> str:
        """Returns the string representation of the ImageBase instance."""
    #     return f'{self.__class__.__name__} shape={"x".join([str(d) for d in self.shape])}'
        return f"BioImage{self.as_tensor().__repr__()[6:]}"

In [7]:
a = BioImage.create('data_examples/example_tiff.tiff')
print(a.shape)





torch.Size([1, 96, 512, 512])


### BioImageStack

class for 3D images

In [8]:
#| export

class BioImageStack(BioImageBase):
    """Subclass of BioImageBase that represents a 3D image object."""
    
    def __repr__(self) -> str:
        """Returns the string representation of the ImageBase instance."""
        return f"BioImageStack{self.as_tensor().__repr__()[6:]}"

In [9]:
a = BioImageStack.create('data_examples/example_tiff.tiff')
print(a.shape)

torch.Size([1, 96, 512, 512])


### BioImageProject

2D representations of 3D stack using maximum intensity projection

In [10]:
#| export

class BioImageProject(BioImageBase):
    """Subclass of BioImageBase that represents a 2D image object."""
    _show_args = {'cmap':'gray'}
    
    @classmethod
    def create(cls, fn: (Path, str, L, list, torchTensor), **kwargs) -> torchTensor: 
        """
        Opens an image and casts it to BioImageBase object.
        If `fn` is a torchTensor, it's cast to BioImageBase object.

        Args:
            fn : (Path, str, torchTensor)
                Image path or a 4D torchTensor.
            kwargs : dict
                Additional parameters for the medical image reader.

        Returns:
            torchTensor : A 3D tensor as a BioImage object.
        """
        if isinstance(fn, torchTensor):
            return cls(fn)

        img = img_reader(fn, dtype=cls, resample=cls.resample, reorder=cls.reorder)
        return torchmax(img, dim=1)[0]  # Taking the maximum intensity projection along axis 1
    
    def show(self, ctx=None, **kwargs):
        "Show image using `merge(self._show_args, kwargs)`"
        return show_image(self, ctx=ctx, **merge(self._show_args, kwargs))
    
    def __repr__(self) -> str:
        """Returns the string representation of the ImageBase instance."""
        return f"BioImage{self.as_tensor().__repr__()[6:]}"

In [11]:
a = BioImageProject.create('data_examples/example_tiff.tiff')
#a = BioImageProject.create('../../bioMONAI_0/_data/Thunder_20230308/nuevos_datos/dataset_2/inputs/1.tif')
a.shape

torch.Size([1, 512, 512])

### BioImageMulti

Multichannel datasets

In [12]:
#| export

class BioImageMulti(BioImageBase):
    """Subclass of BioImageBase that represents a multi-channel 2D image object."""
    
    @classmethod
    def create(cls, fn: (Path, str, L, list, torchTensor), **kwargs) -> torchTensor: 
        """
        Opens an image and casts it to BioImageBase object.
        If `fn` is a torchTensor, it's cast to BioImageBase object.

        Args:
            fn : (Path, str, torchTensor)
                Image path or a 4D torchTensor.
            kwargs : dict
                Additional parameters for the medical image reader.

        Returns:
            torchTensor : A 3D tensor as a BioImage object.
        """
        if isinstance(fn, torchTensor):
            return cls(fn)

        return torchsqueeze(img_reader(fn, dtype=cls, resample=cls.resample, reorder=cls.reorder), 0)
    
    def __repr__(self) -> str:
        """Returns the string representation of the ImageBase instance."""
        return f"BioImageMulti{self.as_tensor().__repr__()[6:]}"
        

In [13]:
# a = BioImageMulti.create('../../bioMONAI_0/_data/Babesia/RI/O11_RI_frame01.tiff')#'../_data/Babesia/RI/O11_RI_frame01.tiff'
# print(a.shape)

### BioImage4D

4D datasets

In [14]:
# TO DO

class BioImage4D(BioImageBase):
    """Subclass of BioImageBase that represents a (multi-channel) 3D image object."""
    
    @classmethod
    def create(cls, fn: (Path, str, L, list, torchTensor), **kwargs) -> torchTensor: 
        """
        Opens an image and casts it to BioImageBase object.
        If `fn` is a torchTensor, it's cast to BioImageBase object.

        Args:
            fn : (Path, str, torchTensor)
                Image path or a 4D torchTensor.
            kwargs : dict
                Additional parameters for the medical image reader.

        Returns:
            torchTensor : A 3D tensor as a BioImage object.
        """
        if isinstance(fn, torchTensor):
            return cls(fn)

        return torchsqueeze(img_reader(fn, dtype=cls, resample=cls.resample, reorder=cls.reorder), 1)
    
    def __repr__(self) -> str:
        """Returns the string representation of the ImageBase instance."""
        return f"BioImage4D{self.as_tensor().__repr__()[6:]}"

In [15]:
# a = BioImage4D.create('../../bioMONAI_0/_data/Babesia/RI/O11_RI_frame01.tiff')#'../_data/Babesia/RI/O11_RI_frame01.tiff'
# print(a.shape)

from torch import randn as torchrandn

tensor = torchrandn(3,10, 4, 5)
b = BioImage4D.create(tensor)#'../_data/Babesia/RI/O11_RI_frame01.tiff'
print(b.shape)

torch.Size([3, 10, 4, 5])


### Data conversion

In [16]:
#| export

class Tensor2BioImage(DisplayedTransform):
    def __init__(self, cls:BioImageBase=BioImageStack):
        self.cls = cls

    def encodes(self, o):
        if isinstance(o, MetaTensor):
            return self.cls(o.clone(), affine=o.affine, meta=o.meta)
        
        if isinstance(o, torchTensor):
            return self.cls(o)

## Data blocks

### ImageBlock

In [17]:
#| export

def BioImageBlock(cls:BioImageBase=BioImageStack):
    "A `TransformBlock` for images of `cls`"
    return TransformBlock(type_tfms=cls.create, batch_tfms=[Tensor2BioImage(cls)]) # IntToFloatTensor

## Data getters

Get the ground truth images located in a folder called 'gt' and divided in labeled subfolders.

> These function are tailor-made for some test datasets, eventually they must be changed/adapted to more general cases


In [18]:
#| export
def get_gt(path, gt_file_name="avg50.png"): 
    def _fn(fn): return Path(path/"gt")/f"{parent_label(fn)}"/gt_file_name
    return _fn


In [19]:
#| export
def get_target(path, same_filename=True, target_file_prefix="target", signal_file_prefix="signal"):
    # Define a function to construct the target file name based on input parameters
    def construct_target_filename(file_name):
        # Split the file name based on the signal file prefix
        parts = file_name.split(signal_file_prefix)
        
        # Construct the target file name by inserting the target file prefix
        target_file_name = parts[0] + target_file_prefix + parts[1]
        
        return target_file_name
    
    # Define a function to generate the target file path based on the given file name
    def generate_target_path(file_name):
        # Extract the base file name
        base_filename = os.path.basename(file_name)
        
        # If same_filename is True, simply return the path joined with the base file name
        if same_filename:
            return Path(path) / base_filename
        
        # If same_filename is False, construct the target file name and return the path joined with it
        target_filename = construct_target_filename(base_filename)
        return Path(path) / target_filename
    
    # Return the appropriate function based on the value of same_filename
    return generate_target_path


In [20]:
print(get_target('train_folder/target', same_filename=False)('../signal/signal01.tif'))
print(get_target('train_folder/target')('../signal/image01.tif'))

train_folder/target/target01.tif
train_folder/target/image01.tif


In [21]:
print(get_target('train_folder', same_filename=False, target_file_prefix="cameraman_clean", signal_file_prefix="cameraman_noisy")('../signal/cameraman_noisy.tif'))

train_folder/cameraman_clean.tif


Get as ground truth another noisy image randomly chosen in the same folder as the input image


In [22]:
#| export
def get_noisy_pair(fn):
    tmp = get_image_files(fn.parent, recurse=False)
    fn2 = tmp[randint(0,len(tmp)-1)]
    while fn2 == fn: fn2 = tmp[randint(0,len(tmp)-1)]
    return fn2

## Data Display

### Show batch

In [23]:
#| export 

@typedispatch
def show_batch(x:BioImageBase, y:BioImageBase, samples, ctxs=None, max_n=10, nrows=None, ncols=None, figsize=None, **kwargs):
    if ctxs is None: ctxs = get_grid(min(len(samples), max_n), nrows=nrows, ncols=ncols, figsize=figsize, double=True)
    for i in range(2):
        ctxs[i::2] = [b.show(ctx=c, **kwargs) for b,c,_ in zip(samples.itemgot(i),ctxs[i::2],range(max_n))]
    return ctxs

### Show results

In [24]:
#| export 

@typedispatch
def show_results(x:BioImageBase, y:BioImageBase, samples, outs, ctxs=None, max_n=10, figsize=None, **kwargs):
    if ctxs is None: ctxs = get_grid(3*min(len(samples), max_n), ncols=3, figsize=figsize, title='Input/Target/Prediction')
    for i in range(2):
        ctxs[i::3] = [b.show(ctx=c, **kwargs) for b,c,_ in zip(samples.itemgot(i),ctxs[i::3],range(max_n))]
    ctxs[2::3] = [b.show(ctx=c, **kwargs) for b,c,_ in zip(outs.itemgot(0),ctxs[2::3],range(max_n))]
    return ctxs

## Example

In [25]:
# from monai.transforms import ScaleIntensity
# from bioMONAI.transforms import *
# from fastai.vision.all import *

In [26]:
# spatial_dimensions = 2
# roi_size = [32]*spatial_dimensions
# item_tfms = [RandCropND(roi_size), ScaleIntensity] 

In [27]:
# file_folder = '../../bioMONAI_0/_data/Thunder_20230308/nuevos_datos/dataset_2'

# dblock = DataBlock(blocks=(BioImageBlock(cls=BioImageProject), BioImageBlock(cls=BioImageProject)),
#                    get_items=get_image_files,
#                    get_y=get_target('../../bioMONAI_0/_data/Thunder_20230308/nuevos_datos/dataset_2/targets', same_filename=True),
#                    splitter=RandomSplitter(valid_pct=0.2),
#                    item_tfms=item_tfms,
#                    )
# # dblock.summary(file_folder)

# dls = dblock.dataloaders(file_folder, bs=2)
# dls.show_batch(max_n=2, cmap='viridis')

---

In [28]:
#| hide
import nbdev; nbdev.nbdev_export()